# 03. DistilBERT 파인튜닝

`distilbert-base-uncased`를 Steam 리뷰 긍/부정 분류로 파인튜닝한다.

산출물: `models/distilbert/` (모델+토크나이저), `models/distilbert/metrics.json`

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import json
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score
from src.config import MODEL_DIR, MAX_LEN, DATA_DIR

BASE_MODEL = "distilbert-base-uncased"
EPOCHS = 2

/Users/gomuseo/Desktop/Python/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL)

# 16GB 통합메모리 MPS OOM 방지: 학습은 max_length 128로 제한 (리뷰 대부분 짧음)
TRAIN_MAX_LEN = 128

def load(split):
    return Dataset.from_pandas(pd.read_csv(DATA_DIR / f"{split}.csv"))

def enc(b):
    return tok(b["text"], truncation=True, max_length=TRAIN_MAX_LEN)

ds = {s: load(s).map(enc, batched=True) for s in ["train", "val", "test"]}
print({s: len(d) for s, d in ds.items()})

Map:   0%|          | 0/11657 [00:00<?, ? examples/s]

Map:  26%|██▌       | 3000/11657 [00:00<00:00, 22534.55 examples/s]

Map:  60%|██████    | 7000/11657 [00:00<00:00, 26774.07 examples/s]

Map:  94%|█████████▍| 11000/11657 [00:00<00:00, 26768.37 examples/s]

Map: 100%|██████████| 11657/11657 [00:00<00:00, 25673.91 examples/s]

...

Map: 100%|██████████| 2498/2498 [00:00<00:00, 10450.75 examples/s]

Map:   0%|          | 0/2498 [00:00<?, ? examples/s]

Map: 100%|██████████| 2498/2498 [00:00<00:00, 19283.34 examples/s]

Map: 100%|██████████| 2498/2498 [00:00<00:00, 18863.36 examples/s]

{'train': 11657, 'val': 2498, 'test': 2498}


In [3]:
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)

def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "f1": f1_score(p.label_ids, preds)}

out = MODEL_DIR / "distilbert"
# 16GB 통합메모리에서 OOM 방지: 배치 8 + 누적 2 (유효 배치 16 유지)
args = TrainingArguments(
    output_dir=str(out), num_train_epochs=EPOCHS,
    per_device_train_batch_size=8, gradient_accumulation_steps=2,
    per_device_eval_batch_size=16,
    eval_strategy="epoch", save_strategy="no", report_to=[])
trainer = Trainer(model=model, args=args,
                  train_dataset=ds["train"], eval_dataset=ds["val"],
                  data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=metrics)
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6481.29it/s]


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


/Users/gomuseo/Desktop/Python/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.617742,0.241213,0.906725,0.944378
2,0.424432,0.310770,0.914732,0.948810


/Users/gomuseo/Desktop/Python/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=1458, training_loss=0.46472511239176095, metrics={'train_runtime': 695.6558, 'train_samples_per_second': 33.514, 'train_steps_per_second': 2.096, 'total_flos': 725876843868144.0, 'train_loss': 0.46472511239176095, 'epoch': 2.0})

In [4]:
test_metrics = trainer.evaluate(ds["test"])
out.mkdir(parents=True, exist_ok=True)
model.save_pretrained(out)
tok.save_pretrained(out)
result = {"accuracy": test_metrics["eval_accuracy"], "f1": test_metrics["eval_f1"]}
(out / "metrics.json").write_text(json.dumps(result, indent=2))
print("DistilBERT test:", result)

/Users/gomuseo/Desktop/Python/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.424432,0.337875,2,0.906325,0.944126


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.59it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.56it/s]

DistilBERT test: {'accuracy': 0.9063250600480385, 'f1': 0.9441260744985673}
